In [ ]:
import warnings
from pathlib import Path

import xarray as xr
import numpy as np

from imagematerials.buildings.preprocessing.floorspace import (
    compute_average_m2_capita,
    compute_housing_residential,
    compute_housing_type,
    extrapolate_floorspace,
    get_image_floorspace
)
from imagematerials.buildings.preprocessing.lifetimes import compute_lifetimes
from imagematerials.buildings.preprocessing.materials import (
    compute_mat_intensities_commercial,
    compute_mat_intensities_residential,
)
from imagematerials.buildings.preprocessing.population import compute_population
from imagematerials.concepts import create_building_graph



from matplotlib import pyplot as plt

from imagematerials.buildings.constants import quintiles_generic

In [ ]:
base_directory = Path("..", "data", "raw")
scenario_sel = "SSP2_baseline"

database_directory = base_directory / "buildings" / "SSP2_CP"
image_directory = base_directory / "image" / scenario_sel

In [ ]:
population = compute_population(image_directory)

In [ ]:
population.sum("Region").sel(Area='Urban').sum("Quintile").plot(label = "Total Urban")
population.sum("Region").sel(Area='Rural').sum("Quintile").plot(label="Total Rural")

for quintile in quintiles_generic:
    population.sum("Region").sel(Area='Urban').sel(Quintile = quintile).plot(label = f"Urban {quintile}")
for quintile in quintiles_generic:
    population.sum("Region").sel(Area='Rural').sel(Quintile = quintile).plot(label = f"Rural {quintile}")
plt.legend()

In [ ]:
# Get floorspace for commercial + urban/rural
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    floorspace_residential_capita, floorspace_commercial_capita, minimum_comm = get_image_floorspace(image_directory, base_directory)
    
floorspace_residential_capita = extrapolate_floorspace(floorspace_residential_capita)
floorspace_commercial_capita = extrapolate_floorspace(floorspace_commercial_capita, minimum_comm)

In [ ]:
for quintile in quintiles_generic:
    floorspace_residential_capita.sel(Region = "4", Area = "Urban").sel(Quintile = quintile).plot(label = f"Residential Urban {quintile}")
    floorspace_residential_capita.sel(Region = "4", Area = "Rural").sel(Quintile = quintile).plot(label = f"Residential Urban {quintile}")
    floorspace_commercial_capita.sel(Region = "4").sum("Type").plot(label = f"Commercial {quintile}")

plt.legend()

In [ ]:
# Average square meter per capita split by residential type [Region, Area, Type]
from imagematerials.buildings.constants import area_graph
average_m2_capita = compute_average_m2_capita(base_directory)

housing_type = compute_housing_type(database_directory)

In [ ]:
# Floorspace m2 for residential buildings [Year, Region, Area, Type]
floorspace_residential = compute_housing_residential(population, 
                                                     average_m2_capita, 
                                                     housing_type, 
                                                     floorspace_residential_capita, {"CE":None})

# Commercial floorspace also needs to be multiplied by population & drop Area dimension
floorspace_commercial_total = floorspace_commercial_capita * population.sel(Area=['Urban', 'Rural']).sum(["Area"]).loc[1721:]

In [ ]:
# Residential housing type shares [Year, Region, Area, Type]
housing_type = compute_housing_type(database_directory)
for res_type in housing_type.coords["Type"].values:
    housing_type.mean(["Region", "Area", "Quintile"]).sel(Type=res_type).plot(label=res_type)
plt.title("Residential housing shares per type.")
plt.legend()
plt.show()

In [ ]:
for res_type in floorspace_residential.coords["Type"].values:
    floorspace_residential.sum(["Region", "Quintile"]).sel(Type=res_type).plot(label=res_type)
for com_type in floorspace_commercial_total.coords["Type"].values:
    floorspace_commercial_total.sum(["Region", "Quintile"]).sel(Type=com_type).plot(label=com_type)
plt.title("Residential housing million m2 per type.")
plt.legend()
plt.show()

In [ ]:
circular_economy_config = {"CE": None}

floorspace = xr.concat((floorspace_residential, floorspace_commercial_total), dim="Type")

# Lifetime computations, see lifetimes.py

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lifetimes = compute_lifetimes(base_directory, floorspace_commercial_total.coords["Type"].values, circular_economy_config)

mat_intensities_comm = compute_mat_intensities_commercial(database_directory, circular_economy_config)
mat_intensities_res = compute_mat_intensities_residential(database_directory, circular_economy_config)
mat_intensities = xr.concat((mat_intensities_res, mat_intensities_comm), dim="Type")
knowledge_graph = create_building_graph()
mat_intensities = knowledge_graph.rebroadcast_xarray(
                        mat_intensities, floorspace.coords["Type"].values)

#TODO remove this quick fix
region_coords = np.sort(floorspace.coords["Region"].values.astype(int)).astype(str)

In [ ]:
mat_intensities